# Import libraries

In [1]:
!pip install lightgbm

In [11]:
import pandas as pd
import numpy as np

from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from lightgbm import LGBMClassifier

import joblib

# Load dataset

In [4]:
df = pd.read_csv(r"C:\Users\priyanka\Desktop\Uploads\model_table1.csv")

# Preprocessing

In [5]:
df = df.dropna(subset=['claim_status'])

df['billing_date'] = pd.to_datetime(df['billing_date'])

le = LabelEncoder()

categorical_cols = ['gender','city','insurance_provider','department','visit_type']

for col in categorical_cols:
    df[col] = le.fit_transform(df[col].astype(str))

# FEATURE ENGINEERING

In [7]:
df['approval_ratio'] = df['approved_amount'] / df['billed_amount']
df['cost_per_day'] = df['billed_amount'] / (df['length_of_stay_hours'] + 1)

df['high_cost_flag'] = (df['billed_amount'] > df['billed_amount'].median()).astype(int)

df.fillna(0, inplace=True)

In [8]:
df['is_long_stay'] = (df['stay_days'] > df['stay_days'].median()).astype(int)

# DEFINE TARGET & FEATURES

In [9]:
y = df['claim_status']

features = [
    'age','gender','city','insurance_provider',
    'billed_amount','approved_amount',
    'length_of_stay_hours',
    'approval_ratio','cost_per_day',
    'visit_frequency','provider_rejection_rate',
    'high_cost_flag'
]

X = df[features]

# TIME-BASED SPLIT

In [10]:
df = df.sort_values(by=['patient_id','billing_date'])

train_idx = []
test_idx = []

for pid, group in df.groupby('patient_id'):
    
    n = len(group)
    split = int(0.8 * n)
    
    train_idx.extend(group.index[:split])
    test_idx.extend(group.index[split:])

X_train = X.loc[train_idx]
X_test = X.loc[test_idx]

y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

# HANDLE CLASS IMBALANCE 

In [12]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print(pd.Series(y_train_res).value_counts())

claim_status
Pending     10812
Paid        10812
Rejected    10812
Name: count, dtype: int64


In [ ]:
#classes = np.unique(y_train)
#weights = compute_class_weight('balanced', classes=classes, y=y_train)

#class_weights = dict(zip(classes, weights))

#print("Class Weights:", class_weights)

Class Weights: {'Paid': 0.5561104945122703, 'Pending': 1.3346651868294488, 'Rejected': 2.209726816121524}


# MODEL

In [13]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

c:\Users\priyanka\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [14]:
rf = RandomForestClassifier(class_weight='balanced', n_estimators=200)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

In [15]:
lgbm = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    class_weight='balanced'
)

lgbm.fit(X_train, y_train)

y_pred_lgbm = lgbm.predict(X_test)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.071750 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1400
[LightGBM] [Info] Number of data points in the train set: 18038, number of used features: 12
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spl

# EVALUATION

In [16]:
print("🔹 Logistic Regression")
print(confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

print("\n🔹 Random Forest")
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

print("\n🔥 LightGBM")
print(confusion_matrix(y_test, y_pred_lgbm))
print(classification_report(y_test, y_pred_lgbm))

🔹 Logistic Regression
[[3899    0  229]
 [1087  596   75]
 [   0    0 1076]]
              precision    recall  f1-score   support

        Paid       0.78      0.94      0.86      4128
     Pending       1.00      0.34      0.51      1758
    Rejected       0.78      1.00      0.88      1076

    accuracy                           0.80      6962
   macro avg       0.85      0.76      0.75      6962
weighted avg       0.84      0.80      0.77      6962


🔹 Random Forest
[[3942    1  185]
 [  10 1683   65]
 [  44    2 1030]]
              precision    recall  f1-score   support

        Paid       0.99      0.95      0.97      4128
     Pending       1.00      0.96      0.98      1758
    Rejected       0.80      0.96      0.87      1076

    accuracy                           0.96      6962
   macro avg       0.93      0.96      0.94      6962
weighted avg       0.96      0.96      0.96      6962


🔥 LightGBM
[[3922    4  202]
 [   4 1685   69]
 [  10    7 1059]]
              precisio

## SAVE BEST MODEL

In [17]:
joblib.dump(lgbm, "claim_model.pkl")

['claim_model.pkl']

In [18]:
df['claim_status'].value_counts(normalize=True) * 100

claim_status
Paid        59.760
Pending     25.052
Rejected    15.188
Name: proportion, dtype: float64

The dataset shows class imbalance, with Paid claims forming the majority (~59%), while Rejected claims constitute only ~16%. This imbalance can bias models toward predicting the majority class, potentially reducing the ability to correctly identify rejected claims, which are critical for financial risk management.

The baseline Logistic Regression model struggled with minority classes, particularly Pending claims, achieving low recall (0.33). This indicates bias toward the majority class. In contrast, ensemble models handled class imbalance more effectively.

# MITIGATION STRATEGY

To address class imbalance, class weights were applied during model training. This assigns higher importance to minority classes, ensuring that misclassification of rejected and pending claims is penalized more heavily during learning.

After applying class balancing techniques, the model achieved high recall for minority classes, particularly Rejected claims (0.98 recall), which is critical for minimizing financial losses. This demonstrates the effectiveness of the imbalance handling strategy.

High recall for rejected claims ensures that potentially risky claims are identified early, allowing hospitals to take corrective actions before submission. This can significantly reduce revenue leakage and improve financial efficiency